# LeetCode 1716 — Calculate Money in Leetcode Bank bằng Python

**Link đề:** https://leetcode.com/problems/calculate-money-in-leetcode-bank/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def totalMoney(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Mô phỏng từng ngày với hai thông tin: **đang ở tuần thứ mấy** và **thứ mấy trong tuần**.
2. Dùng thành thạo `//` (tuần) và `%` (ngày trong tuần) để đổi "ngày thứ `i`" thành vị trí trong lịch.
3. Xây công thức **cấp số cộng**: tổng `1 + 2 + ... + m = m(m+1)/2`, tổng mỗi tuần tăng đều 7.
4. Đi từ mô phỏng `O(n)` → cộng theo tuần `O(n/7)` → **công thức đóng `O(1)`**.
5. Kiểm chứng công thức bằng đối chiếu với mô phỏng trên mọi `n` trong constraints.

## 1. Đề bài và ví dụ

Hercy gửi tiền vào ngân hàng **mỗi ngày**:
- Thứ Hai đầu tiên (ngày 1) gửi `$1`.
- Từ thứ Ba đến Chủ nhật, mỗi ngày gửi **nhiều hơn hôm trước `$1`**.
- Mỗi thứ Hai tiếp theo gửi **nhiều hơn thứ Hai tuần trước `$1`**.

Cho `n`, trả về **tổng số tiền** sau ngày thứ `n`.

| `n` | Diễn giải | Kết quả |
|---:|---|---:|
| `4` | 1 + 2 + 3 + 4 | `10` |
| `10` | (1+…+7) + (2+3+4) | `37` |
| `20` | (1+…+7) + (2+…+8) + (3+…+8) | `96` |

**Constraints:** `1 <= n <= 1000`.

### 1.1. Lập bảng lịch bằng tay

| | T2 | T3 | T4 | T5 | T6 | T7 | CN | tổng tuần |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Tuần 1 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 28 |
| Tuần 2 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 35 |
| Tuần 3 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 42 |

Quan sát: mỗi tuần sau **dịch lên 1** so với tuần trước ⇒ tổng tuần tăng thêm **7**.

In [1]:
for week in range(3):
    row = [week + d for d in range(1, 8)]
    print(f"Tuần {week + 1}: {row} -> tổng = {sum(row)}")

Tuần 1: [1, 2, 3, 4, 5, 6, 7] -> tổng = 28
Tuần 2: [2, 3, 4, 5, 6, 7, 8] -> tổng = 35
Tuần 3: [3, 4, 5, 6, 7, 8, 9] -> tổng = 42


## 2. Nền tảng Python cần biết

### 2.1. `//` và `%` để đổi ngày thành (tuần, thứ)

Đánh số ngày từ `0`: ngày `i` (0-based) thuộc tuần `i // 7` và là ngày thứ `i % 7` trong tuần (0 = thứ Hai). Số tiền ngày đó = `(i // 7) + (i % 7) + 1`.

In [2]:
for i in [0, 1, 6, 7, 8, 13, 14]:
    week, dow = i // 7, i % 7
    print(f"ngày (0-based) {i:>2}: tuần {week}, thứ {dow} trong tuần -> gửi ${week + dow + 1}")

ngày (0-based)  0: tuần 0, thứ 0 trong tuần -> gửi $1
ngày (0-based)  1: tuần 0, thứ 1 trong tuần -> gửi $2
ngày (0-based)  6: tuần 0, thứ 6 trong tuần -> gửi $7
ngày (0-based)  7: tuần 1, thứ 0 trong tuần -> gửi $2
ngày (0-based)  8: tuần 1, thứ 1 trong tuần -> gửi $3
ngày (0-based) 13: tuần 1, thứ 6 trong tuần -> gửi $8
ngày (0-based) 14: tuần 2, thứ 0 trong tuần -> gửi $3


### 2.2. `divmod` — lấy tuần đầy đủ và số ngày lẻ

`w, r = divmod(n, 7)`: `w` = số **tuần trọn vẹn**, `r` = số **ngày lẻ** của tuần cuối (0..6).

In [3]:
for n in [4, 7, 10, 20]:
    w, r = divmod(n, 7)
    print(f"n = {n:>2}: {w} tuần đầy đủ + {r} ngày lẻ")

n =  4: 0 tuần đầy đủ + 4 ngày lẻ
n =  7: 1 tuần đầy đủ + 0 ngày lẻ
n = 10: 1 tuần đầy đủ + 3 ngày lẻ
n = 20: 2 tuần đầy đủ + 6 ngày lẻ


### 2.3. Tổng cấp số cộng `1 + 2 + ... + m = m(m + 1) / 2`

Mẹo Gauss: ghép số đầu với số cuối. `1 + 7 = 2 + 6 = 3 + 5 = 8`… Có `m` số, mỗi cặp tổng `m + 1`, có `m/2` cặp ⇒ `m(m+1)/2`. Tổng quát: tổng cấp số cộng = **(số hạng đầu + số hạng cuối) × số số hạng / 2**.

Trong Python dùng `//` để giữ kết quả là `int` (tích `m(m+1)` luôn chẵn nên chia đúng).

In [4]:
for m in [4, 7, 10]:
    print(f"1..{m}: sum(range) = {sum(range(1, m + 1))}, công thức = {m * (m + 1) // 2}")
print("2 + 3 + ... + 8 =", sum(range(2, 9)), "| (2 + 8) * 7 // 2 =", (2 + 8) * 7 // 2)

1..4: sum(range) = 10, công thức = 10
1..7: sum(range) = 28, công thức = 28
1..10: sum(range) = 55, công thức = 55
2 + 3 + ... + 8 = 35 | (2 + 8) * 7 // 2 = 35


### 2.4. `range(start, stop)` và `sum`

`range(1, 8)` sinh `1..7` (không gồm 8). `sum(iterable)` cộng tất cả. Đây là công cụ cho lời giải mô phỏng.

In [5]:
print(list(range(1, 8)), sum(range(1, 8)))
print(list(range(2, 5)), sum(range(2, 5)))

[1, 2, 3, 4, 5, 6, 7] 28
[2, 3, 4] 9


## 3. Cách trực giác — mô phỏng từng ngày

Giữ `monday` (số tiền thứ Hai của tuần hiện tại) và `today` (số tiền hôm nay):
- Ngày đầu tuần: `today = monday`.
- Các ngày sau: `today += 1`.
- Hết tuần (sau Chủ nhật): `monday += 1`.

In [6]:
def total_money_simulation(n: int) -> int:
    total = 0
    monday = 1
    today = 0
    for day in range(n):            # day = 0, 1, ..., n-1
        if day % 7 == 0:            # đầu tuần
            today = monday
            monday += 1             # chuẩn bị cho thứ Hai tuần sau
        else:
            today += 1
        total += today
    return total

print(total_money_simulation(4), total_money_simulation(10), total_money_simulation(20))   # 10 37 96

10 37 96


### Gọn hơn: tính thẳng số tiền mỗi ngày

Từ mục 2.1: ngày `i` gửi `i // 7 + i % 7 + 1`.

In [7]:
def total_money_direct(n: int) -> int:
    return sum(i // 7 + i % 7 + 1 for i in range(n))

print(total_money_direct(4), total_money_direct(10), total_money_direct(20))

10 37 96


### Chi phí và hướng cải tiến

Mô phỏng: `O(n)` thời gian, `O(1)` bộ nhớ. Với `n ≤ 1000` đã rất nhanh. Nhưng bài thuộc chủ đề **Math** — có cấu trúc lặp lại theo tuần:
- Cả tuần được cộng **theo khối** → không cần đi từng ngày.
- Tổng các tuần tạo **cấp số cộng** (28, 35, 42, …) → không cần đi từng tuần.

## 4. Insight — tách thành "tuần đầy đủ" + "ngày lẻ"

Gọi `w, r = divmod(n, 7)`.

**Phần 1 — `w` tuần đầy đủ:** tuần thứ `k` (`k = 0..w-1`) có tổng `28 + 7k`. Đây là cấp số cộng `w` số hạng, đầu `28`, cuối `28 + 7(w-1)`:
`full = (28 + 28 + 7(w - 1)) · w / 2 = 28w + 7·w(w-1)/2`.

**Phần 2 — `r` ngày lẻ của tuần thứ `w + 1`:** thứ Hai tuần này gửi `w + 1`, các ngày là `w+1, w+2, ..., w+r`:
`partial = r·w + (1 + 2 + ... + r) = r·w + r(r+1)/2`.

**Đáp án:** `full + partial`.

In [8]:
n = 20
w, r = divmod(n, 7)
full = 28 * w + 7 * w * (w - 1) // 2
partial = r * w + r * (r + 1) // 2
print(f"n = {n}: w = {w}, r = {r}")
print(f"  phần tuần đầy đủ = 28*{w} + 7*{w}*{w - 1}/2 = {full}   (28 + 35 = 63)")
print(f"  phần ngày lẻ     = {r}*{w} + {r}*{r + 1}/2 = {partial}   (3+4+5+6+7+8 = 33)")
print("  tổng =", full + partial)

n = 20: w = 2, r = 6
  phần tuần đầy đủ = 28*2 + 7*2*1/2 = 63   (28 + 35 = 63)
  phần ngày lẻ     = 6*2 + 6*7/2 = 33   (3+4+5+6+7+8 = 33)
  tổng = 96


## 5. Lời giải chính — công thức đóng `O(1)`

Ghép lại thành hàm. Mọi phép chia đều dùng `//`: `w(w-1)` và `r(r+1)` là tích hai số liên tiếp nên luôn chẵn ⇒ chia 2 chính xác.

In [9]:
def total_money_formula(n: int) -> int:
    w, r = divmod(n, 7)
    full = 28 * w + 7 * w * (w - 1) // 2
    partial = r * w + r * (r + 1) // 2
    return full + partial

print(total_money_formula(4), total_money_formula(10), total_money_formula(20))   # 10 37 96

10 37 96


## 6. TRACE từng bước

### 6.1. Trace mô phỏng với `n = 10`

| ngày (1-based) | day % 7 | thứ | today | monday sau | total |
|---:|---:|---|---:|---:|---:|
| 1 | 0 | T2 | 1 | 2 | 1 |
| 2 | 1 | T3 | 2 | 2 | 3 |
| … | … | … | … | … | … |
| 7 | 6 | CN | 7 | 2 | 28 |
| 8 | 0 | T2 | 2 | 3 | 30 |
| 9 | 1 | T3 | 3 | 3 | 33 |
| 10 | 2 | T4 | 4 | 3 | 37 |

In [10]:
DAY_NAMES = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]

def trace_simulation(n: int) -> int:
    total, monday, today = 0, 1, 0
    for day in range(n):
        if day % 7 == 0:
            today = monday
            monday += 1
        else:
            today += 1
        total += today
        print(f"ngày {day + 1:>2} ({DAY_NAMES[day % 7]}): gửi {today:>2}, monday kế tiếp = {monday}, total = {total}")
    return total

trace_simulation(10)

ngày  1 (T2): gửi  1, monday kế tiếp = 2, total = 1
ngày  2 (T3): gửi  2, monday kế tiếp = 2, total = 3
ngày  3 (T4): gửi  3, monday kế tiếp = 2, total = 6
ngày  4 (T5): gửi  4, monday kế tiếp = 2, total = 10
ngày  5 (T6): gửi  5, monday kế tiếp = 2, total = 15
ngày  6 (T7): gửi  6, monday kế tiếp = 2, total = 21
ngày  7 (CN): gửi  7, monday kế tiếp = 2, total = 28
ngày  8 (T2): gửi  2, monday kế tiếp = 3, total = 30
ngày  9 (T3): gửi  3, monday kế tiếp = 3, total = 33
ngày 10 (T4): gửi  4, monday kế tiếp = 3, total = 37


37

### 6.2. Trace công thức theo từng tuần với `n = 20`

In [11]:
def trace_by_week(n: int) -> int:
    w, r = divmod(n, 7)
    total = 0
    for k in range(w):
        week_sum = 28 + 7 * k
        total += week_sum
        print(f"tuần {k + 1} (đầy đủ): {list(range(k + 1, k + 8))} -> {week_sum}, total = {total}")
    if r:
        days = list(range(w + 1, w + 1 + r))
        total += sum(days)
        print(f"tuần {w + 1} ({r} ngày lẻ): {days} -> {sum(days)}, total = {total}")
    return total

trace_by_week(20)

tuần 1 (đầy đủ): [1, 2, 3, 4, 5, 6, 7] -> 28, total = 28
tuần 2 (đầy đủ): [2, 3, 4, 5, 6, 7, 8] -> 35, total = 63
tuần 3 (6 ngày lẻ): [3, 4, 5, 6, 7, 8] -> 33, total = 96


96

## 7. Các lời giải khác

### 7.1. Cộng theo tuần (vòng lặp `O(n / 7)`)

Trung gian giữa mô phỏng và công thức: lặp qua từng tuần đầy đủ (cộng `28 + 7k`), rồi cộng ngày lẻ.

In [12]:
def total_money_weeks(n: int) -> int:
    w, r = divmod(n, 7)
    total = 0
    for k in range(w):
        total += 28 + 7 * k
    for d in range(1, r + 1):
        total += w + d
    return total

print(total_money_weeks(4), total_money_weeks(10), total_money_weeks(20))

10 37 96


### 7.2. Lồng hai vòng `for` — duyệt lịch như bảng

Vòng ngoài là tuần, vòng trong là ngày; dừng khi đủ `n` ngày. Dễ hình dung nhất cho người mới.

In [13]:
def total_money_nested(n: int) -> int:
    total, days = 0, 0
    week = 0
    while days < n:
        for d in range(7):
            if days == n:
                break
            total += week + d + 1
            days += 1
        week += 1
    return total

print(total_money_nested(4), total_money_nested(10), total_money_nested(20))

10 37 96


### 7.3. Công thức "tổng trung bình × số số hạng" cho tuần đầy đủ

Viết lại phần tuần đầy đủ: trung bình mỗi ngày của tuần `k` là `k + 4` (giữa `k+1` và `k+7`) ⇒ tổng tuần = `7(k + 4)`. Tổng `w` tuần = `7·(4w + w(w-1)/2)`. Cùng kết quả, chỉ là cách nhìn khác.

In [14]:
def total_money_avg(n: int) -> int:
    w, r = divmod(n, 7)
    return 7 * (4 * w + w * (w - 1) // 2) + r * w + r * (r + 1) // 2

print([total_money_avg(n) for n in [4, 10, 20]])

[10, 37, 96]


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Mô phỏng `monday/today` | đúng như đề | `O(n)` | `O(1)` | dễ nghĩ nhất |
| Công thức ngày `i//7 + i%7 + 1` | mỗi ngày một công thức | `O(n)` | `O(1)` | gọn |
| Lồng vòng tuần/ngày | duyệt bảng lịch | `O(n)` | `O(1)` | trực quan |
| Cộng theo tuần | `28 + 7k` | `O(n/7)` | `O(1)` | bước đệm |
| **Công thức đóng** | cấp số cộng | `O(1)` | `O(1)` | tối ưu |

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def totalMoney(self, n: int) -> int:
        w, r = divmod(n, 7)
        return 28 * w + 7 * w * (w - 1) // 2 + r * w + r * (r + 1) // 2

sol = Solution()
print(sol.totalMoney(4), sol.totalMoney(10), sol.totalMoney(20))

10 37 96


Bản mô phỏng (an toàn, dễ giải thích khi phỏng vấn):

In [16]:
class SolutionSimulation:
    def totalMoney(self, n: int) -> int:
        return sum(i // 7 + i % 7 + 1 for i in range(n))

print(SolutionSimulation().totalMoney(20))

96


## 9. Vì sao đúng?

**Số tiền ngày `i` (0-based) là `i // 7 + i % 7 + 1`:**
- Ngày `i` thuộc tuần `t = i // 7` (0-based) và là ngày `d = i % 7` trong tuần (0 = thứ Hai).
- Thứ Hai tuần `t` gửi `t + 1` (tuần 0 gửi 1, mỗi tuần thêm 1). Ngày `d` trong tuần gửi nhiều hơn thứ Hai `d` đồng ⇒ `t + 1 + d`. ✓

**Công thức đóng:** tách `n = 7w + r` (`0 ≤ r < 7`).
- Tuần `k` đầy đủ: `Σ_{d=0..6} (k + 1 + d) = 7k + 28`. Tổng `k = 0..w-1`: `28w + 7·(0 + 1 + ... + (w-1)) = 28w + 7·w(w-1)/2`.
- Tuần thứ `w` (0-based) chỉ có `r` ngày: `Σ_{d=0..r-1} (w + 1 + d) = r·w + r(r+1)/2`.

Mô phỏng `monday/today` đúng vì nó duy trì invariant: sau khi xử lý ngày `i`, `today = i // 7 + i % 7 + 1` và `monday = i // 7 + 2` (thứ Hai **kế tiếp**).

## 10. Độ phức tạp

- **Công thức:** vài phép nhân/chia ⇒ **Time `O(1)`, Space `O(1)`**.
- **Mô phỏng:** `n` vòng ⇒ `O(n)`, Space `O(1)`.
- **Cộng theo tuần:** `w + r ≤ n/7 + 6` vòng ⇒ `O(n)` (hằng số nhỏ hơn 7 lần).

Kết quả lớn nhất khi `n = 1000`: vài chục nghìn — nằm thoải mái trong `int` (Python không lo tràn số).

In [17]:
print("n = 1000 ->", total_money_formula(1000))
import time
t = time.perf_counter(); [total_money_simulation(n) for n in range(1, 1001)]; t1 = time.perf_counter() - t
t = time.perf_counter(); [total_money_formula(n) for n in range(1, 1001)]; t2 = time.perf_counter() - t
print(f"1000 lần mô phỏng: {t1:.4f}s | 1000 lần công thức: {t2:.5f}s")

n = 1000 -> 74926
1000 lần mô phỏng: 0.0308s | 1000 lần công thức: 0.00028s


## 11. Bộ test

1. Ba ví dụ đề: `4 → 10`, `10 → 37`, `20 → 96`.
2. Biên: `n = 1` → 1; `n = 7` (đúng một tuần) → 28; `n = 8` (sang thứ Hai mới) → 30; `n = 14` → 63; `n = 1000`.
3. Đối chiếu **mọi** cách với **mọi** `n` từ 1 đến 1000 (toàn bộ constraints!).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề n=4", S.totalMoney(4), 10)
check("đề n=10", S.totalMoney(10), 37)
check("đề n=20", S.totalMoney(20), 96)
check("n=1", S.totalMoney(1), 1)
check("n=7 (1 tuần)", S.totalMoney(7), 28)
check("n=8 (thứ Hai mới)", S.totalMoney(8), 30)
check("n=14 (2 tuần)", S.totalMoney(14), 63)

PASS đề n=4: 10
PASS đề n=10: 37
PASS đề n=20: 96
PASS n=1: 1
PASS n=7 (1 tuần): 28
PASS n=8 (thứ Hai mới): 30
PASS n=14 (2 tuần): 63


In [19]:
impls = [total_money_simulation, total_money_direct, total_money_formula, total_money_weeks,
         total_money_nested, total_money_avg, Solution().totalMoney, SolutionSimulation().totalMoney]
for n in range(1, 1001):
    vals = {f(n) for f in impls}
    assert len(vals) == 1, (n, vals)
print("8 cài đặt khớp nhau với mọi n = 1..1000: PASS")

8 cài đặt khớp nhau với mọi n = 1..1000: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Thứ Hai mới "nối tiếp" Chủ nhật (gửi 8 thay vì 2)

In [20]:
def wrong_continue(n):
    return sum(range(1, n + 1))        # SAI: coi như mỗi ngày +1 mãi mãi

print("sai:", wrong_continue(10), "| đúng:", total_money_formula(10))

sai: 55 | đúng: 37


**Sửa:** thứ Hai tuần mới = thứ Hai tuần trước **+ 1**, không phải Chủ nhật + 1.

### Lỗi 2 — Đánh số ngày từ 1 nhưng vẫn dùng `% 7 == 0` làm thứ Hai

Nếu `day` chạy `1..n`, thì `day % 7 == 0` là **Chủ nhật**, không phải thứ Hai → lệch toàn bộ.

In [21]:
def wrong_one_based(n):
    total, monday, today = 0, 1, 0
    for day in range(1, n + 1):        # 1-based
        if day % 7 == 0:               # SAI: ngày 7 mới là thứ Hai?!
            today = monday; monday += 1
        else:
            today += 1
        total += today
    return total

print("sai:", wrong_one_based(10), "| đúng:", total_money_formula(10))

sai: 31 | đúng: 37


**Sửa:** dùng `range(n)` (0-based) với `day % 7 == 0`, hoặc 1-based với `(day - 1) % 7 == 0`.

### Lỗi 3 — Dùng `/` trong công thức → kết quả `float`

In [22]:
def wrong_float(n):
    w, r = divmod(n, 7)
    return 28 * w + 7 * w * (w - 1) / 2 + r * w + r * (r + 1) / 2   # SAI: trả float

print("sai:", wrong_float(20), type(wrong_float(20)), "| đúng:", total_money_formula(20), type(total_money_formula(20)))

sai: 96.0 <class 'float'> | đúng: 96 <class 'int'>


Giá trị đúng nhưng kiểu `float` (`96.0`). LeetCode có thể vẫn chấp nhận, nhưng với số lớn `float` mất chính xác. **Sửa:** dùng `//` (an toàn vì tích hai số liên tiếp luôn chẵn).

### Lỗi 4 — Quên ngày lẻ khi `r > 0`, hoặc tính tuần lẻ bắt đầu từ `1` thay vì `w + 1`

In [23]:
def wrong_partial(n):
    w, r = divmod(n, 7)
    return 28 * w + 7 * w * (w - 1) // 2 + r * (r + 1) // 2    # SAI: thiếu r * w

print("sai:", wrong_partial(20), "| đúng:", total_money_formula(20))

sai: 84 | đúng: 96


**Sửa:** ngày lẻ của tuần thứ `w + 1` bắt đầu từ `w + 1` ⇒ cộng thêm `r * w`.

## 13. Mẹo nhận dạng pattern

- Dữ liệu **lặp theo chu kỳ** (tuần 7 ngày, tháng, vòng tròn) → `divmod(n, chu_kỳ)` tách "số chu kỳ đầy đủ" + "phần dư".
- Giá trị tăng đều mỗi bước → **cấp số cộng**: tổng = (đầu + cuối) × số số hạng / 2.
- Bài Math với `n` nhỏ → luôn viết **mô phỏng trước** để có "đáp án chuẩn", rồi đối chiếu công thức trên **toàn bộ** miền `n`.
- Bài liên quan: 1688 Count of Matches in Tournament, 1523 Count Odd Numbers in an Interval Range (công thức `//`), 441 Arranging Coins (tổng `1..k`), 2177 Find Three Consecutive Integers That Sum to a Given Number.

## 14. Biến thể

### Biến thể 1 — Chu kỳ `p` ngày thay vì 7, mỗi chu kỳ mới tăng `step`

In [24]:
def total_money_general(n: int, p: int = 7, step: int = 1) -> int:
    w, r = divmod(n, p)
    base = p * (p + 1) // 2                 # tổng chu kỳ đầu tiên: 1..p
    full = base * w + p * step * w * (w - 1) // 2
    partial = r * (r + 1) // 2 + r * step * w
    return full + partial

def sim_general(n, p=7, step=1):
    return sum((i // p) * step + i % p + 1 for i in range(n))

for n, p, step in [(20, 7, 1), (20, 5, 2), (13, 3, 10)]:
    print(f"n={n}, p={p}, step={step}: công thức = {total_money_general(n, p, step)}, mô phỏng = {sim_general(n, p, step)}")

n=20, p=7, step=1: công thức = 96, mô phỏng = 96
n=20, p=5, step=2: công thức = 120, mô phỏng = 120
n=13, p=3, step=10: công thức = 245, mô phỏng = 245


### Biến thể 2 — Ngày đầu tiên tìm được số tiền ≥ `target`

Ngược lại bài gốc: tìm `n` nhỏ nhất sao cho `totalMoney(n) >= target`. Vì tổng **tăng ngặt** theo `n`, có thể dùng **tìm kiếm nhị phân**.

In [25]:
def first_day_reaching(target: int) -> int:
    lo, hi = 1, 1
    while total_money_formula(hi) < target:   # tìm cận trên bằng cách nhân đôi
        hi *= 2
    while lo < hi:                             # nhị phân trên [lo, hi]
        mid = (lo + hi) // 2
        if total_money_formula(mid) >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo

print(first_day_reaching(37), first_day_reaching(38), first_day_reaching(10**6))

10 11 3718


### Biến thể 3 — Số tiền gửi **trong ngày** `n` (không phải tổng)

In [26]:
def deposit_on_day(n: int) -> int:
    i = n - 1
    return i // 7 + i % 7 + 1

print([deposit_on_day(n) for n in range(1, 16)])

[1, 2, 3, 4, 5, 6, 7, 2, 3, 4, 5, 6, 7, 8, 3]


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Tính tay
`n = 15`: có bao nhiêu tuần đầy đủ, bao nhiêu ngày lẻ, tổng là bao nhiêu?

In [27]:
def exercise_1():
    # TODO: trả về (w, r, tổng)
    pass

# assert exercise_1() == (2, 1, Solution().totalMoney(15))

### Bài 2 — Điền chỗ trống mô phỏng

In [28]:
def simulate_fill(n: int) -> int:
    total = 0
    # TODO: duyệt i trong range(n), cộng số tiền của ngày i (dùng // và %)
    pass

# assert simulate_fill(10) == 37

### Bài 3 — Tổng của riêng tuần thứ `k` (1-based, đầy đủ 7 ngày)

In [29]:
def week_sum(k: int) -> int:
    # TODO: O(1)
    pass

# assert week_sum(1) == 28 and week_sum(2) == 35 and week_sum(3) == 42

### Bài 4 — Tự viết lại công thức đóng

In [30]:
def formula_rewrite(n: int) -> int:
    # TODO: divmod + hai cấp số cộng
    pass

# assert all(formula_rewrite(n) == total_money_simulation(n) for n in range(1, 200))

### Bài 5 — Biến thể: chỉ gửi tiền từ thứ Hai đến thứ Sáu
Thứ Bảy và Chủ nhật **không gửi** (nhưng lịch vẫn tính bình thường). Tổng sau `n` ngày?

In [31]:
def total_weekdays_only(n: int) -> int:
    # TODO
    pass

# assert total_weekdays_only(7) == 15 and total_weekdays_only(10) == 15 + 2 + 3 + 4

### Bài 6 — Mini challenge: tiền gửi theo tuần tăng gấp đôi
Thứ Hai tuần `k` (0-based) gửi `2^k`, các ngày sau trong tuần vẫn +1 mỗi ngày. Viết công thức `O(log n)` hoặc `O(n/7)`, đối chiếu với mô phỏng.

In [32]:
def total_doubling(n: int) -> int:
    # TODO
    pass

# assert total_doubling(10) == (1+2+3+4+5+6+7) + (2+3+4)
# assert total_doubling(16) == 28 + (2+3+4+5+6+7+8) + (4+5)

## 16. Gợi ý 3 mức

**Bài 1:** (1) `divmod(15, 7)`. (2) `w = 2, r = 1`. (3) `28 + 35 + 3 = 66`.

**Bài 2:** (1) Công thức ngày: `i // 7 + i % 7 + 1`. (2) `total += ...`. (3) `return total`.

**Bài 3:** (1) Tuần `k` bắt đầu bằng `k`. (2) Các ngày `k, k+1, ..., k+6`. (3) `7k + 21`.

**Bài 4:** (1) `w, r = divmod(n, 7)`. (2) Tuần đầy đủ: `28w + 7w(w-1)/2`. (3) Ngày lẻ: `rw + r(r+1)/2`.

**Bài 5:** (1) Bỏ những ngày có `i % 7 >= 5`. (2) Tuần `k` chỉ còn `5k + 15`. (3) Mô phỏng: `sum(... for i in range(n) if i % 7 < 5)`.

**Bài 6:** (1) Thứ Hai tuần `k` là `2**k`. (2) Ngày `i` gửi `2**(i//7) + i % 7`. (3) Cộng theo tuần: tuần đầy đủ `7·2^k + 21`.

## 17. Đáp án tham khảo

In [33]:
def ans_exercise_1():
    w, r = divmod(15, 7)
    return (w, r, 28 + 35 + 3)

def ans_simulate_fill(n):
    total = 0
    for i in range(n):
        total += i // 7 + i % 7 + 1
    return total

def ans_week_sum(k):
    return 7 * k + 21

def ans_formula_rewrite(n):
    w, r = divmod(n, 7)
    return 28 * w + 7 * w * (w - 1) // 2 + r * w + r * (r + 1) // 2

def ans_total_weekdays_only(n):
    return sum(i // 7 + i % 7 + 1 for i in range(n) if i % 7 < 5)

def ans_total_doubling(n):
    w, r = divmod(n, 7)
    total = sum(7 * 2 ** k + 21 for k in range(w))
    total += sum(2 ** w + d for d in range(r))
    return total

assert ans_exercise_1() == (2, 1, Solution().totalMoney(15))
assert ans_simulate_fill(10) == 37
assert ans_week_sum(1) == 28 and ans_week_sum(2) == 35 and ans_week_sum(3) == 42
assert all(ans_formula_rewrite(n) == total_money_simulation(n) for n in range(1, 200))
assert ans_total_weekdays_only(7) == 15 and ans_total_weekdays_only(10) == 15 + 2 + 3 + 4
assert ans_total_doubling(10) == (1 + 2 + 3 + 4 + 5 + 6 + 7) + (2 + 3 + 4)
assert ans_total_doubling(16) == 28 + (2 + 3 + 4 + 5 + 6 + 7 + 8) + (4 + 5)
assert all(ans_total_doubling(n) == sum(2 ** (i // 7) + i % 7 for i in range(n)) for n in range(1, 100))
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def totalMoney(self, n: int) -> int:
        w, r = divmod(n, 7)                      # w tuần đầy đủ, r ngày lẻ
        full = 28 * w + 7 * w * (w - 1) // 2     # 28, 35, 42, ... (cấp số cộng)
        partial = r * w + r * (r + 1) // 2       # (w+1) + ... + (w+r)
        return full + partial
```

Dự phòng mô phỏng: `sum(i // 7 + i % 7 + 1 for i in range(n))`.

**Khung tư duy:** chu kỳ → `divmod` → cấp số cộng cho phần đầy đủ + phần dư → đối chiếu với mô phỏng.

**Checklist trước khi submit**
- [ ] Thứ Hai mới = thứ Hai cũ + 1 (không nối tiếp Chủ nhật)?
- [ ] Đánh số ngày 0-based khi dùng `% 7 == 0` làm thứ Hai?
- [ ] Dùng `//` để kết quả là `int`?
- [ ] Đã thử `n = 1, 7, 8, 14`?
- [ ] Tên method `totalMoney`, trả về `int`?